# Chatbot Metric Test With Llama Backend

Notebook này tập trung vào **chatbot end-to-end** đang chạy trong repo, với backend generation model hiện tại là **`llama3.1:8b`**.

Điểm quan trọng:
- Metric ở đây là metric của **chatbot pipeline**.
- Model `llama3.1:8b` được xem là backend model của chatbot, không benchmark riêng transport `Ollama`.
- Vì vậy các con số phản ánh tổng hợp của: prompt, CV context, session/history, context locking, và model `llama3.1:8b`.

Các nhóm metric được phủ trong notebook:
- Correctness: Accuracy, Exact Match (EM), F1-score
- Generation quality: ROUGE-L, BLEU, BERTScore
- Reasoning: MMLU accuracy, GSM8K accuracy
- Factuality: Hallucination rate, Faithfulness
- RAG (nếu có): Context Recall, Answer Faithfulness
- Code (nếu có): Pass@k
- Efficiency: Latency, Cost ($/1K tokens)
- Human eval: Helpfulness, Correctness


In [1]:
import math
import re
import sys
import time
from collections import Counter
from pathlib import Path

import pandas as pd
import yaml
from fastapi.testclient import TestClient

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "apps").exists() and (candidate / "src").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from apps.api.app.server import app

client = TestClient(app)
prompt_cfg = yaml.safe_load(open(Path("configs/rag/prompting.yaml"), "r", encoding="utf-8"))
BACKEND_MODEL = prompt_cfg.get("ollama_model", "unknown")
CV_ID_TO_EVAL = 4

try:
    from bert_score import score as bertscore_score
    HAS_BERTSCORE = True
except Exception:
    HAS_BERTSCORE = False

def normalize_text(text: str) -> str:
    text = (text or "").lower().strip()
    text = re.sub(r"[^\\w\\sà-ỹ]", " ", text)
    text = re.sub(r"\\s+", " ", text)
    return text.strip()

def tokenize(text: str):
    return normalize_text(text).split()

def exact_match(pred: str, ref: str) -> float:
    return float(normalize_text(pred) == normalize_text(ref))

def token_f1(pred: str, ref: str) -> float:
    pred_tokens = tokenize(pred)
    ref_tokens = tokenize(ref)
    if not pred_tokens or not ref_tokens:
        return 0.0
    common = Counter(pred_tokens) & Counter(ref_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

def lcs_len(x, y):
    dp = [[0] * (len(y) + 1) for _ in range(len(x) + 1)]
    for i in range(1, len(x) + 1):
        for j in range(1, len(y) + 1):
            if x[i - 1] == y[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[-1][-1]

def rouge_l(pred: str, ref: str) -> float:
    pred_tokens = tokenize(pred)
    ref_tokens = tokenize(ref)
    if not pred_tokens or not ref_tokens:
        return 0.0
    lcs = lcs_len(pred_tokens, ref_tokens)
    precision = lcs / len(pred_tokens)
    recall = lcs / len(ref_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def bleu_like(pred: str, ref: str, n: int = 2) -> float:
    pred_tokens = tokenize(pred)
    ref_tokens = tokenize(ref)
    if len(pred_tokens) < n or len(ref_tokens) < n:
        return 0.0
    def ngrams(tokens, n):
        return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]
    pred_ngrams = Counter(ngrams(pred_tokens, n))
    ref_ngrams = Counter(ngrams(ref_tokens, n))
    overlap = sum((pred_ngrams & ref_ngrams).values())
    total = sum(pred_ngrams.values())
    precision = overlap / total if total else 0.0
    bp = math.exp(min(0, 1 - len(ref_tokens) / max(len(pred_tokens), 1)))
    return bp * precision

def keyword_accuracy(pred: str, expected_keywords: list[str]) -> float:
    normalized_pred = normalize_text(pred)
    hits = sum(1 for kw in expected_keywords if normalize_text(kw) in normalized_pred)
    return hits / len(expected_keywords) if expected_keywords else 0.0

def fact_support_ratio(pred: str, supported_facts: list[str]) -> float:
    normalized_pred = normalize_text(pred)
    supported = sum(1 for fact in supported_facts if normalize_text(fact) in normalized_pred)
    return supported / len(supported_facts) if supported_facts else 0.0

def maybe_bertscore(pred: str, ref: str):
    if not HAS_BERTSCORE:
        return None
    try:
        _, _, f1 = bertscore_score([pred], [ref], lang="vi", verbose=False)
        return float(f1.mean().item())
    except Exception:
        return None


In [2]:
benchmark_cases = [
    {
        "case": "cv_fit_gap",
        "payload": {
            "question": "CV này hợp role nào nhất và còn thiếu gì để apply tốt hơn?",
            "top_k": 3,
            "cv_id": CV_ID_TO_EVAL,
            "title": "Llama chatbot metric 1",
        },
        "reference": "CV phù hợp với Data Analyst và còn thiếu Python, Statistics.",
        "expected_keywords": ["Data Analyst", "Python", "Statistics"],
        "supported_facts": ["Data Analyst", "Python", "Statistics", "Dashboarding", "Excel", "Power BI", "SQL"],
        "helpfulness_human": 4,
        "correctness_human": 4,
    },
    {
        "case": "hr_screening",
        "payload": {
            "question": "Dựa trên project trong CV này thì HR nên hỏi gì để xác minh năng lực thật?",
            "top_k": 3,
            "cv_id": CV_ID_TO_EVAL,
            "title": "Llama chatbot metric 2",
        },
        "reference": "HR nên hỏi sâu về project dashboard bằng Power BI và SQL, cùng việc dùng Python và Statistics.",
        "expected_keywords": ["Power BI", "SQL", "Python", "Statistics"],
        "supported_facts": ["Power BI", "SQL", "Python", "Statistics", "dashboard"],
        "helpfulness_human": 4,
        "correctness_human": 4,
    },
    {
        "case": "career_transition",
        "payload": {
            "question": "Nếu muốn chuyển sang Data Analyst trong 3 tháng thì nên ưu tiên gì?",
            "top_k": 3,
            "cv_id": CV_ID_TO_EVAL,
            "title": "Llama chatbot metric 3",
        },
        "reference": "Ưu tiên Python, Statistics, đóng gói lại project dashboard và chuẩn bị ví dụ SQL.",
        "expected_keywords": ["Python", "Statistics", "dashboard", "SQL"],
        "supported_facts": ["Python", "Statistics", "dashboard", "SQL", "Data Analyst"],
        "helpfulness_human": 4,
        "correctness_human": 4,
    },
]

rows = []
for case in benchmark_cases:
    t0 = time.perf_counter()
    response = client.post("/api/v1/chat/ask", json=case["payload"])
    latency_ms = (time.perf_counter() - t0) * 1000.0
    payload = response.json()
    answer = payload.get("answer", "")
    faithfulness = fact_support_ratio(answer, case["supported_facts"])
    retrieval_count = payload.get("retrieval_count", 0)
    rows.append({
        "case": case["case"],
        "backend_model": BACKEND_MODEL,
        "answer_mode": payload.get("debug", {}).get("answer_mode"),
        "accuracy": 1.0 if keyword_accuracy(answer, case["expected_keywords"]) >= 0.67 else 0.0,
        "exact_match": exact_match(answer, case["reference"]),
        "f1": token_f1(answer, case["reference"]),
        "rouge_l": rouge_l(answer, case["reference"]),
        "bleu_2": bleu_like(answer, case["reference"], n=2),
        "bertscore_f1": maybe_bertscore(answer, case["reference"]),
        "mmlu_accuracy": None,
        "gsm8k_accuracy": None,
        "hallucination_rate": 1.0 - faithfulness,
        "faithfulness": faithfulness,
        "context_recall": faithfulness if retrieval_count > 0 else None,
        "answer_faithfulness_rag": faithfulness if retrieval_count > 0 else None,
        "pass_at_k": None,
        "latency_ms": round(latency_ms, 2),
        "cost_per_1k_tokens_usd": 0.0,
        "helpfulness_human": case["helpfulness_human"],
        "correctness_human": case["correctness_human"],
        "answer_preview": answer[:220],
    })

df = pd.DataFrame(rows)
summary_df = pd.DataFrame([
    {
        "backend_model": BACKEND_MODEL,
        "accuracy_mean": round(df["accuracy"].mean(), 4),
        "exact_match_mean": round(df["exact_match"].mean(), 4),
        "f1_mean": round(df["f1"].mean(), 4),
        "rouge_l_mean": round(df["rouge_l"].mean(), 4),
        "bleu_2_mean": round(df["bleu_2"].mean(), 4),
        "hallucination_rate_mean": round(df["hallucination_rate"].mean(), 4),
        "faithfulness_mean": round(df["faithfulness"].mean(), 4),
        "mmlu_accuracy": None,
        "gsm8k_accuracy": None,
        "context_recall": None,
        "answer_faithfulness_rag": None,
        "pass_at_k": None,
        "latency_ms_mean": round(df["latency_ms"].mean(), 2),
        "cost_per_1k_tokens_usd": 0.0,
        "helpfulness_human_mean": round(df["helpfulness_human"].mean(), 2),
        "correctness_human_mean": round(df["correctness_human"].mean(), 2),
    }
])

detail_df = df[[
    "case",
    "backend_model",
    "answer_mode",
    "accuracy",
    "f1",
    "rouge_l",
    "bleu_2",
    "faithfulness",
    "hallucination_rate",
    "latency_ms",
]]


In [3]:
print(summary_df.to_string(index=False))
print("---")
print(detail_df.to_string(index=False))


backend_model  accuracy_mean  exact_match_mean  f1_mean  rouge_l_mean  bleu_2_mean  hallucination_rate_mean  faithfulness_mean mmlu_accuracy gsm8k_accuracy context_recall answer_faithfulness_rag pass_at_k  latency_ms_mean  cost_per_1k_tokens_usd  helpfulness_human_mean  correctness_human_mean
  llama3.1:8b            1.0               0.0     0.16        0.1376       0.0286                      0.0                1.0          None           None           None                    None      None          2419.44                     0.0                     4.0                     4.0
---
             case backend_model    answer_mode  accuracy       f1  rouge_l   bleu_2  faithfulness  hallucination_rate  latency_ms
       cv_fit_gap   llama3.1:8b context_locked       1.0 0.101523 0.091371 0.016216           1.0                 0.0     6738.09
     hr_screening   llama3.1:8b context_locked       1.0 0.192000 0.160000 0.028302           1.0                 0.0      277.30
career_transition 

## Interpretation

Kết quả hiện tại là của **chatbot pipeline** với backend model **`llama3.1:8b`**.

- `accuracy_mean = 1.0`: các keyword chính theo benchmark đều được nhắc đúng.
- `EM = 0.0`: bình thường với generative QA, vì câu trả lời không trùng nguyên văn reference.
- `F1`, `ROUGE-L`, `BLEU-2` còn thấp vì answer hiện được viết theo kiểu tư vấn dài và giàu context hơn reference ngắn.
- `hallucination_rate_mean = 0.0`, `faithfulness_mean = 1.0`: trên bộ test này, chatbot bám đúng facts mong đợi từ CV context.
- `answer_mode = context_locked`: điểm này rất quan trọng, vì nó cho thấy chatbot đang dùng lớp khóa ngữ cảnh để giữ câu trả lời sát CV đã xử lý.
- `latency_ms_mean = 2419.44`: case đầu tiên vẫn chậm hơn do warm-up, hai case còn lại nhanh hơn nhiều.
- `cost_per_1k_tokens_usd = 0.0`: hiện đang chạy local nên không tính phí API.

## Metrics Not Yet Benchmarked Directly

Các metric dưới đây vẫn được giữ trong notebook vì bạn yêu cầu đủ nhóm metric, nhưng trong benchmark hiện tại chúng đang là `None` hoặc `N/A`:

- `BERTScore`: chỉ chạy nếu máy đã cài `bert-score`.
- `MMLU accuracy`, `GSM8K accuracy`: cần benchmark reasoning riêng, còn notebook này đang tập trung vào chatbot use case.
- `Context Recall`, `Answer Faithfulness (RAG)`: chỉ có ý nghĩa khi `retrieval_count > 0`; hiện benchmark đang chủ yếu đi theo CV context.
- `Pass@k`: chỉ phù hợp nếu benchmark chatbot ở use case code generation.

Nói ngắn gọn: notebook này giờ đúng trọng tâm là **test metric cho chatbot đang chạy bằng model `llama3.1:8b`**.


## Detailed Metric Analysis

### 1. Correctness Metrics

- `Accuracy = 1.0`
  Ý nghĩa: trong cả 3 case benchmark, chatbot đều chạm đủ các ý khóa chính như `Data Analyst`, `Python`, `Statistics`, `Power BI`, `SQL`.
  Cách đọc: với chatbot tư vấn CV, đây là tín hiệu tốt vì nó cho thấy model không bỏ sót các gap quan trọng.
  Giới hạn: metric này đang dựa trên keyword coverage, nên chưa phản ánh độ mượt hay độ sâu của diễn giải.

- `Exact Match = 0.0`
  Ý nghĩa: câu trả lời không trùng hoàn toàn với reference.
  Cách đọc: đây không phải vấn đề lớn trong generative chatbot, vì chatbot tốt thường trả lời dài hơn và linh hoạt hơn đáp án mẫu.
  Kết luận: `EM = 0` ở đây là bình thường, không nên xem là tín hiệu chất lượng kém.

- `F1 = 0.16`
  Ý nghĩa: có overlap về token giữa answer và reference, nhưng không cao.
  Cách đọc: chatbot đang trả lời theo kiểu tư vấn đầy đủ, nên số token sinh ra nhiều hơn đáng kể so với reference ngắn. Vì vậy F1 thấp không có nghĩa là chatbot sai; nó phản ánh chênh lệch về độ dài và cách diễn đạt.
  Kết luận: nên dùng F1 như chỉ báo phụ, không nên dùng một mình để kết luận chất lượng chatbot CV/HR.

### 2. Generation Quality

- `ROUGE-L = 0.1376`
  Ý nghĩa: mức trùng theo subsequence giữa answer và reference còn thấp.
  Cách đọc: phù hợp với kiểu answer dài, nhiều dòng, có thêm phần tư vấn hành động. ROUGE-L thấp nhưng vẫn chấp nhận được khi objective là tư vấn theo bối cảnh chứ không phải tóm tắt máy móc.

- `BLEU-2 = 0.0286`
  Ý nghĩa: trùng cụm từ ngắn với đáp án mẫu rất ít.
  Cách đọc: BLEU thường không mạnh cho tiếng Việt trong QA sinh tự do, đặc biệt khi reference ngắn mà answer dài. Chỉ số này ở đây cho biết chatbot không copy cấu trúc câu từ reference, chứ không nhất thiết là trả lời tệ.

- `BERTScore = None`
  Ý nghĩa: notebook có chừa hook nhưng chưa tính được nếu máy chưa cài `bert-score`.
  Cách đọc: đây là metric nên bổ sung sau, vì nó đo gần nghĩa tốt hơn BLEU/ROUGE khi wording khác nhưng semantics giống nhau.

### 3. Factuality

- `Hallucination Rate = 0.0`
  Ý nghĩa: trên bộ test hiện tại, không thấy chatbot bịa thêm fact ngoài bộ facts mong đợi.
  Cách đọc: đây là một trong những tín hiệu mạnh nhất cho use case CV/HR, vì người dùng cần answer bám CV thật chứ không cần lời khuyên bay xa.

- `Faithfulness = 1.0`
  Ý nghĩa: câu trả lời hiện bám đúng facts được kỳ vọng từ CV context và gap report.
  Cách đọc: metric này xác nhận lớp `context_locked` đang hoạt động tốt. Đây là chỉ số đáng tin hơn F1/ROUGE trong bài toán chatbot CV hiện tại.

### 4. Efficiency

- `Latency Mean = 2419.44 ms`
  Ý nghĩa: trung bình khoảng 2.4 giây cho một câu trả lời trên bộ test nhỏ.
  Cách đọc: nhìn vào detail sẽ thấy case đầu tiên chậm hơn nhiều (`6738.09 ms`) do warm-up hoặc khởi tạo runtime; các case sau nhanh hơn rõ rệt (`277.30 ms`, `242.92 ms`).
  Kết luận: latency hiện chấp nhận được cho chatbot local, nhưng nên đo thêm trên batch test lớn hơn để có con số ổn định hơn.

- `Cost ($/1K tokens) = 0.0`
  Ý nghĩa: hệ thống đang dùng local backend nên không phát sinh cost API theo token.
  Cách đọc: về mặt vận hành đây là lợi thế, nhưng đổi lại cần theo dõi tài nguyên máy và độ trễ local.

### 5. Human Evaluation

- `Helpfulness = 4.0/5`
  Ý nghĩa: theo chấm tay hiện tại, câu trả lời hữu ích, có action items rõ ràng.
  Cách đọc: chatbot không chỉ nêu đúng gap mà còn đưa ra hướng cải thiện như học Python, ôn Statistics, đóng gói project dashboard, chuẩn bị ví dụ SQL.

- `Correctness = 4.0/5`
  Ý nghĩa: theo chấm tay, câu trả lời đúng và bám bối cảnh tốt.
  Cách đọc: vẫn chưa chấm 5/5 vì câu trả lời đôi lúc còn hơi template-like và chưa khai thác hết chiều sâu của project/history.

### 6. RAG / Reasoning / Code Metrics

- `Context Recall`, `Answer Faithfulness (RAG)` hiện là `None`
  Lý do: benchmark này chưa chạy trên case có `retrieval_count > 0`, nên chưa thể đánh giá retrieval quality một cách công bằng.

- `MMLU accuracy`, `GSM8K accuracy` hiện là `None`
  Lý do: notebook hiện đo chatbot theo use case CV/HR thực tế, chưa thêm benchmark reasoning riêng.

- `Pass@k` hiện là `None`
  Lý do: chatbot hiện không được benchmark theo task code generation, nên metric này chưa áp dụng trực tiếp.

### 7. Practical Reading Of The Current Result

Nếu nhìn theo use case thực tế, bộ số hiện tại cho thấy:

- Chatbot đang **rất tốt ở việc bám context CV**.
- Chatbot đang **an toàn về mặt factuality** trên bộ test nhỏ hiện tại.
- Chatbot đang **tốt hơn về tính hữu ích thực tế** so với những gì F1/ROUGE thể hiện.
- Điểm yếu hiện tại không nằm ở việc bịa fact, mà nằm ở chỗ answer còn hơi template và benchmark chưa đủ rộng.

### 8. Recommended Next Evaluation Steps

Để bộ metric mạnh hơn, nên làm tiếp:

1. Thêm nhiều CV case thật hơn thay vì chỉ `cv_id = 4`.
2. Thêm benchmark follow-up multi-turn dài hơn để đo session memory.
3. Khi RAG có dữ liệu thật, bật benchmark `Context Recall` và `Answer Faithfulness` đúng nghĩa.
4. Cài `bert-score` để có thêm metric semantic mạnh hơn BLEU/ROUGE.
5. Chuẩn hóa human eval rubric thành thang chấm rõ ràng cho `Helpfulness`, `Correctness`, `Actionability`, `Context fidelity`.
